# M01: Preprocessing

Crop black border → pad square → resize → circular FOV mask → Ben Graham colour normalisation. Tests: every output has the target size, no crash on 200 random images, before/after grid looks consistent. **Not executed in the repository.**

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (cand / "src" / "drscreen").is_dir():
        sys.path.insert(0, str(cand / "src")); ROOT = cand; break
# On Kaggle the package is embedded in the kernel script; for notebooks, upload src/drscreen as a Dataset or copy it next to the notebook.
DATA_ROOT = os.environ.get("DR_DATA_ROOT") or ("/kaggle/input/aptos2019-blindness-detection" if Path("/kaggle/input").exists() else str(ROOT / "data" / "aptos2019"))
print("project root:", ROOT, "| data root:", DATA_ROOT)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, time
from drscreen.datasets import load_labelled_dataset
from drscreen.preprocess import read_image, preprocess_plain, to_model_input, ImageError
df = load_labelled_dataset(DATA_ROOT)
SIZE = 384
sample = df.sample(min(200, len(df)), random_state=0)
bad, t0 = [], time.time()
for r in sample.itertuples():
    try:
        out = preprocess_plain(read_image(r.path), SIZE)
        assert out.shape == (SIZE, SIZE, 3) and out.dtype == np.uint8
    except (ImageError, AssertionError) as e:
        bad.append((r.image_id, str(e)))
print(f'{len(sample)-len(bad)}/{len(sample)} images OK, {len(bad)} failures, {(time.time()-t0)/len(sample)*1000:.0f} ms/image'); print(bad[:5])

In [ ]:
fig, axs = plt.subplots(3, 6, figsize=(15, 8))
for j, r in enumerate(sample.head(6).itertuples()):
    raw = read_image(r.path); plain = preprocess_plain(raw, SIZE)
    for i, (im, t) in enumerate(((raw, 'original'), (plain, 'cropped/masked'), (to_model_input(plain), 'Ben Graham'))):
        axs[i, j].imshow(im); axs[i, j].set_title(f'{t} (grade {r.grade})', fontsize=8); axs[i, j].axis('off')
plt.tight_layout(); plt.show()